# Long Pipeline Surge — Phase 7 Verification

Canonical **LP-SURGE-01** case: **5-mile** sloping transmission main with elevation survey, **grid cap 2000**, **interior DVCM**, and profile export. Mirrors **`tests/test_long_pipeline_surge.py`** (validation LP-02–LP-04).

| Check | Description |
|-------|-------------|
| LP-02 | Static minimum gauge pressure at survey summit |
| LP-03 | Interior cavity at summit under downsurge (not terminals) |
| LP-04 | Cavity collapse secondary spike after downstream refill |

> **Runtime:** two 8 s transients (~15–20 s total). Grid cap yields ~230% wave-speed distortion on this reach (documented tradeoff; see Phase 4 roadmap).

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Flong_pipeline_surge_verification.ipynb)

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")
from long_pipeline_surge_verification_utils import (
    CASE_ID,
    DEFAULT_DT_S,
    DEFAULT_LENGTH_MI,
    SUMMIT_CHAINAGE_FT,
    SUMMIT_ELEVATION_FT,
    default_survey,
    evaluate_collapse_spike,
    evaluate_grid_cap,
    evaluate_summit_cavity,
    evaluate_summit_static,
    run_downsurge_case,
    run_refill_collapse_case,
    run_static_preview,
    summit_index,
)
from rthym_moc.report import summarize_study

## 2. Elevation survey and grid metadata

In [ ]:
survey = default_survey()
static_results = run_static_preview()
grid = evaluate_grid_cap(static_results)
static = evaluate_summit_static(static_results)
print(f"Case {CASE_ID}: {DEFAULT_LENGTH_MI:.0f} mi, N={grid.num_segments}, distortion={grid.distortion_pct:.1f}%")
print(f"Grid cap PASS={grid.passed}  LP-02 static PASS={static.passed}")

chainage = np.asarray(static_results["pipe_profile_chainage_ft"]["Pmain"])
z = [pt[1] for pt in survey]
x = [pt[0] for pt in survey]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x, z, "o-", color="saddlebrown", label="Survey")
ax.axvline(SUMMIT_CHAINAGE_FT, color="gray", linestyle=":", label="Summit chainage")
ax.set_xlabel("Chainage (ft)")
ax.set_ylabel("Ground elevation (ft)")
ax.set_title("LP-SURGE-01 elevation profile")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. Downsurge — interior cavity at summit (LP-03)

In [ ]:
downsurge = run_downsurge_case()
cavity = evaluate_summit_cavity(downsurge)
print(
    f"LP-03: summit max cavity={cavity.summit_max_volume_ft3:.4f} ft³, "
    f"active steps={cavity.summit_active_steps}, PASS={cavity.passed}"
)

t = np.asarray(downsurge["time"])
chainage = np.asarray(downsurge["pipe_profile_chainage_ft"]["Pmain"])
idx = summit_index(chainage)
head = np.asarray(downsurge["pipe_profile_head"]["Pmain"])[:, idx]
vol = np.asarray(downsurge["pipe_profile_cavity_volume"]["Pmain"])[:, idx]
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, head, color="steelblue")
axes[0].set_ylabel("Summit head (ft)")
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, vol, color="darkcyan")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Cavity volume (ft³)")
axes[1].grid(True, alpha=0.3)
fig.suptitle(f"Summit (chainage {chainage[idx]:.0f} ft, z={SUMMIT_ELEVATION_FT:.0f} ft)")
fig.tight_layout()
plt.show()

## 4. Chainage envelope (pressure min/max along line)

In [ ]:
summary = summarize_study(downsurge, dt_s=DEFAULT_DT_S)
env = summary["pipes"]["Pmain"]["chainage_envelope"]
xc = np.asarray(env["chainage_ft"]) / 5280.0
pmin = np.asarray(env["pressure_min_psi"])
pmax = np.asarray(env["pressure_max_psi"])
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(xc, pmin, pmax, alpha=0.25, color="steelblue", label="Min–max envelope")
ax.plot(xc, pmin, color="navy", label="Min gauge P")
ax.axvline(SUMMIT_CHAINAGE_FT / 5280.0, color="gray", linestyle=":", label="Summit")
ax.set_xlabel("Chainage (mi)")
ax.set_ylabel("Gauge pressure (psi)")
ax.set_title("Downsurge pressure envelope along Pmain")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5. Refill-driven collapse spike (LP-04)

In [ ]:
refill = run_refill_collapse_case()
spike = evaluate_collapse_spike(refill)
print(
    f"LP-04: collapse step={spike.collapse_step}, rise={spike.rise_ft:.2f} ft, "
    f"peak above vapor={spike.peak_above_vapor_ft:.2f} ft, PASS={spike.passed}"
)

t = np.asarray(refill["time"])
chainage = np.asarray(refill["pipe_profile_chainage_ft"]["Pmain"])
idx = summit_index(chainage)
head = np.asarray(refill["pipe_profile_head"]["Pmain"])[:, idx]
vol = np.asarray(refill["pipe_profile_cavity_volume"]["Pmain"])[:, idx]
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, head, color="firebrick")
if spike.collapse_step >= 0:
    axes[0].axvline(spike.collapse_step * DEFAULT_DT_S, color="orange", linestyle=":", label="First collapse")
axes[0].set_ylabel("Summit head (ft)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, vol, color="darkcyan")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Cavity volume (ft³)")
axes[1].grid(True, alpha=0.3)
fig.suptitle("Downsurge + downstream refill — collapse at summit")
fig.tight_layout()
plt.show()

## 6. Summary

In [ ]:
overall = grid.passed and static.passed and cavity.passed and spike.passed
print("Overall:", "PASS" if overall else "FAIL")
for label, ok in [
    ("Grid cap / metadata", grid.passed),
    ("LP-02 static summit minimum", static.passed),
    ("LP-03 interior cavity at summit", cavity.passed),
    ("LP-04 collapse secondary spike", spike.passed),
]:
    print(f"  {label}: {'PASS' if ok else 'FAIL'}")